# Variant A — fully nested participant-level evaluation

Only Variant A is trained, never original LPGNet or a GRU baseline. Configuration selection uses three inner participant folds, with a separate early-stopping split inside each inner training set. LP coefficients/scales use **only core training controls**. Configuration and epochs are selected within outer training before fixed-epoch refitting of model and LP on all outer-training participants and one outer-test evaluation.

## Read before running: profile and budget
- Apply `RUN_PROFILE='efficient'` **after** the original configuration and SPACE definitions. It retains **10 outer folds, one seed, three candidates**, caps **100/40**, stopping/LR patience **12/5**, and `CACHE_FROZEN_ENCODER=True`. Three inner assessments per candidate plus epoch selection and refit give **220 training stages / 15,400 maximum epochs**.
- `full` restores the original 12 candidates, seed 42, **1,000/200** caps, patience 25/14 and uncached protocol: 760 stages / 456,000 maximum epochs. A small model is not a small experiment.
- `fixed` uses one configuration and skips redundant inner configuration assessment, retaining training-only epoch selection and final refit: 40 stages / 2,800 maximum epochs. It is not a hyperparameter search.
- The cache accelerates stage 2 with an **inference-mode frozen encoder**: frozen stochastic noise/dropout is disabled, unlike the full original protocol. This is a declared protocol change, **not identical training distributions**.
- Scientific settings are frozen before training. Changing them requires a **fresh kernel and new checkpoint RUN_ID**, not mutating an existing `run_result`. Candidate lists and inner folds are independent of training seed.

## Verified off-runtime recovery
Run the storage cell immediately before the runner even while `RUN_EXPERIMENT=False`. Colab support mounts Drive via its API. Outside Colab, configure a real authenticated **rclone off-runtime remote** (required on Kaggle); never paste credentials into cells. Probe upload/download verification and automatic restore must succeed before training.

Every completed epoch automatically commits model/optimizer, history and stopping/LR state to verified off-runtime storage; inner and outer tasks commit independently. Repeated storage execution reuses the matching mirror/scratch. Losing local scratch is safe after verified commits; a crash mid-epoch can lose at most the current uncommitted epoch. Cloud failures stop training; provider availability and permanent retention are not guaranteed. One runtime/writer per RUN_ID. Leave `RESTORE_FROM_ZIP=''`: old manual ZIP restore must not be mixed with automatic recovery. Optional ZIP export is only for portability.

`MAX_NEW_TASKS=1` bounds each runner invocation. The curated checksum-verified cohort contains 161 participants / 292 recordings; exact-ID clinical joins are diagnostics only. Registered attention pooling and modern Keras APIs support bounded-memory training and safe model serialization. Old CSV-only checkpoints are not reused.

**Source pin:** SOURCE_REV is `52a050399d0330ad6dd76cf5499b7d1520e57f4d`. Checks include durable storage; pin failures must not be bypassed. This update does not execute research training.

In [ ]:
# Run this setup first in a fresh Kaggle/Colab kernel.
if 'run_result' in globals() or 'FROZEN_SCIENTIFIC_PROFILE' in globals():
    raise RuntimeError('Setup would reset a frozen run. Continue with the runner; use a fresh kernel and new RUN_ID for changes.')
import os, sys, tempfile, subprocess
from pathlib import Path
from collections import Counter
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH','true')
SOURCE_REV = '52a050399d0330ad6dd76cf5499b7d1520e57f4d'  # Pinned committed support sources.
REPO_URL = 'https://github.com/AlexM3020/Subject-Independent-Evaluation-of-Mamba-for-Parkinson-s-Disease-Detection-from-Gait-Signals.git'
INSTALL_DEPENDENCIES = False
roots=[Path.cwd(),*Path.cwd().parents]
REPO_ROOT=next((p for b in roots for p in (b,b/'paper_repository') if (p/'gait/tf_notebook_support.py').is_file()),None)
if REPO_ROOT is None:
    REPO_ROOT=Path(tempfile.gettempdir())/('gait-keras-'+SOURCE_REV[:12])
    if not REPO_ROOT.exists(): subprocess.run(['git','clone','--no-checkout',REPO_URL,str(REPO_ROOT)],check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'checkout','--detach',SOURCE_REV],check=True)
REPO_ROOT=REPO_ROOT.resolve()
for filename in ('__init__.py','official_data.py','tf_notebook_support.py','durable_storage.py'):
    expected=subprocess.run(['git','-C',str(REPO_ROOT),'show',SOURCE_REV+':gait/'+filename],capture_output=True,check=True).stdout
    assert (REPO_ROOT/'gait'/filename).read_bytes().replace(b'\r\n',b'\n')==expected.replace(b'\r\n',b'\n'), 'Pinned source changed: '+filename
stale=[name for name,module in list(sys.modules.items()) if (name=='gait' or name.startswith('gait.')) and getattr(module,'__file__',None) and not Path(module.__file__).resolve().is_relative_to(REPO_ROOT)]
if stale: raise RuntimeError('Older gait imports cached: restart kernel and run this setup first. '+str(stale))
sys.path.insert(0,str(REPO_ROOT))
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO_ROOT/'requirements-tensorflow.txt')],check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
keras=tf.keras
from scipy.signal import lfilter
from sklearn.metrics import (roc_auc_score,accuracy_score,balanced_accuracy_score,f1_score,confusion_matrix,ConfusionMatrixDisplay,log_loss,roc_curve)
from IPython.display import display
from gait import tf_notebook_support as nb
FAMILY='A'
SEED,FOLD_SEED,SPLIT_SEED,SEARCH_SEED=1,0,9137,20261117
N_FOLDS,N_INNER,VAL_FRAC=10,3,.15
LP_ORDER,WIN,STRIDE=11,100,50
BATCH_WIN,BATCH_REC=128,64
LR_WIN,LR_REC,LABEL_SMOOTH,L2_REG=5e-4,1e-3,.1,1e-3
MAX_EPOCHS_WIN,MAX_EPOCHS_REC=1000,200
PATIENCE,RLR_PATIENCE,MIN_LR=25,14,1e-7
THRESHOLD=.5
WORK_ROOT=Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content') if Path('/content').exists() else Path.cwd()
# CHECKPOINT_DIR and OUTDIR are assigned by the verified storage cell.
CACHE_DIR=Path(tempfile.gettempdir())/'gaitpdb-1.0.0-verified-cache'
COHORT_POLICY='curated'
ALLOW_DOWNLOAD=True
RUN_EXPERIMENT= True
MAX_NEW_TASKS= 1000
VERBOSE_EPOCHS=1
RESTORE_FROM_ZIP=''  # Legacy only: do not use with automatic durable restore.
EXPERIMENT_COMPLETE=False
summary=participant_ci=participant_fold_metrics=selection_table=trial_table=nested_oof=pd.DataFrame()
participants=participant_errors=pd_severity=by_study=seed_stability=inner_loss=pd.DataFrame()
keras.utils.set_random_seed(SEED)
print('Pinned source:',SOURCE_REV,'| TensorFlow:',tf.__version__,'| Keras:',keras.__version__)
print('GPU:',tf.config.list_physical_devices('GPU'))
print('Select the scientific profile, then verify off-runtime storage before enabling training.')

Pinned source: 52a050399d0330ad6dd76cf5499b7d1520e57f4d | TensorFlow: 2.21.0 | Keras: 3.13.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Select the scientific profile, then verify off-runtime storage before enabling training.


In [ ]:
# Original full-protocol settings; the later profile cell applies the chosen budget AFTER SPACE_A.
TRAIN_SEEDS = [42]
N_TRIALS = 12
N_BOOT = 1000
print('Variant A only; no baseline or GRU training.')
print('These are original configuration values, not the final run budget. See the profile cell before storage.')

Variant A only; no baseline or GRU training.
These are original configuration values, not the final run budget. See the profile cell before storage.


In [ ]:
records,clinical,data_audit,data_provenance = nb.load_inputs(CACHE_DIR,COHORT_POLICY,ALLOW_DOWNLOAD)
subjects=sorted({r['subject'] for r in records})
subj_label={r['subject']:r['label'] for r in records}
by_file={r['file']:i for i,r in enumerate(records)}
by_subject={s:[i for i,r in enumerate(records) if r['subject']==s] for s in subjects}
print('Cohort policy:',COHORT_POLICY)
print(len(records),'recordings |',len(subjects),'participants |',dict(Counter(subj_label.values())))
print('Data provenance:',data_provenance)
print('Clinical unknowns are retained; no severity is inferred from filenames.')

Cohort policy: curated
292 recordings | 161 participants | {0: 71, 1: 90}
Data provenance: {'cohort': {'source': 'https://physionet.org/files/gaitpdb/1.0.0/', 'manifest_sha256': '44903ce52d5428ba218a9cdfd60d5646ab6674df4926233c756ae7c3c7cd50ec', 'min_samples': 2000, 'excluded_subjects': ['GaPt17', 'GaPt18', 'JuCo26', 'JuPt01'], 'conflict_groups': [['GaPt17_02.txt', 'GaPt18_01.txt'], ['JuCo26_01.txt', 'JuPt01_01.txt']], 'policy': 'exclude_all_recordings_of_cross_identity_duplicate_participants'}, 'clinical': {'source': 'https://physionet.org/files/gaitpdb/1.0.0/', 'manifest_sha256': '44903ce52d5428ba218a9cdfd60d5646ab6674df4926233c756ae7c3c7cd50ec', 'files': {'demographics.txt': 'ebe5e3c5c3055023b876225d3eb067a9351693e28ce47fa678872ead8284f643', 'demographics.html': '26871fae0ea56ace5260ed9c31eaeaf1f5d374df53a03ba8493d8308deaf6658'}, 'issues': [{'raw_id': 'Juc010', 'reason': 'invalid_exact_id_or_short_row'}], 'join': 'exact study-qualified ID; TXT/HTML HY and identity cross-check'}}
Cli

In [ ]:
def preprocess(x):
    return lfilter([.5,.5],1,x,axis=0)[::2]

def fit_lp(control_sigs,order=LP_ORDER):
    return nb.fit_lp(control_sigs,order)

def apply_lp(x,C,stds):
    return nb.apply_lp(x,C,stds)

def lp_features(indices,C,stds):
    return {i:apply_lp(records[i]['proc'],C,stds) for i in indices}

for r in records:
    r['proc']=preprocess(r['signal'])
print('Only downsampling is prepared globally. Early-stopping and inner-assessment controls never fit the selection-stage LP reference.')

Only downsampling is prepared globally. Early-stopping and inner-assessment controls never fit the selection-stage LP reference.


In [ ]:
def make_folds():
    return nb.outer_folds(records,N_FOLDS,FOLD_SEED)

def split_val(indices,frac=VAL_FRAC,seed=SEED):
    return nb.subject_split(records,indices,frac,seed)

def labels(indices):
    return np.array([records[i]['label'] for i in indices],np.float32)

FOLDS=make_folds()
for k,f in enumerate(FOLDS):
    print('Fold',k,'train participants:',len({records[i]['subject'] for i in f['train']}),
          'test participants:',len({records[i]['subject'] for i in f['test']}))

Fold 0 train participants: 144 test participants: 17
Fold 1 train participants: 145 test participants: 16
Fold 2 train participants: 145 test participants: 16
Fold 3 train participants: 145 test participants: 16
Fold 4 train participants: 145 test participants: 16
Fold 5 train participants: 145 test participants: 16
Fold 6 train participants: 145 test participants: 16
Fold 7 train participants: 145 test participants: 16
Fold 8 train participants: 145 test participants: 16
Fold 9 train participants: 145 test participants: 16


In [ ]:
# Shared Keras3-compatible training support; no materialization of all training windows.
# Models are checkpointed with optimizer state after each complete epoch.
# EarlyStopping/LR counters and absolute-epoch batch order are preserved.
build_recording_model=nb.build_recording_model
print('Epoch recovery uses committed .keras models plus atomic JSON state, not CSV row counts.')
print('Old CSV-only checkpoint directories are intentionally not imported into this corrected protocol.')

Epoch recovery uses committed .keras models plus atomic JSON state, not CSV row counts.
Old CSV-only checkpoint directories are intentionally not imported into this corrected protocol.


In [ ]:
# ============================================================
# OOF metrics, participant aggregation, errors, and clinical subgroups
# ============================================================
def _validated_binary_inputs(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    if y.ndim != 1 or p.ndim != 1 or len(y) == 0 or len(y) != len(p):
        raise ValueError('Labels and probabilities must be nonempty 1-D arrays of equal length.')
    if np.iscomplexobj(y) or not np.isin(y, [0, 1]).all():
        raise ValueError('Labels must contain only binary values 0 and 1.')
    if np.iscomplexobj(p):
        raise ValueError('Probabilities must be real numbers in [0, 1].')
    p = np.asarray(p, float)
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('Probabilities must be finite and in [0, 1].')
    return y.astype(int), p

def metric_dict(y, p):
    y, p = _validated_binary_inputs(y, p)
    yh = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0,1]).ravel()
    return dict(n=len(y), accuracy=accuracy_score(y, yh),
                balanced_accuracy=balanced_accuracy_score(y, yh),
                macro_f1=f1_score(y, yh, labels=[0,1], average='macro', zero_division=0),
                f1_pd=f1_score(y, yh, zero_division=0),
                sensitivity=tp/(tp+fn) if tp+fn else np.nan,
                specificity=tn/(tn+fp) if tn+fp else np.nan,
                roc_auc=roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
                log_loss=log_loss(y, np.clip(p,1e-7,1-1e-7), labels=[0,1]),
                brier=float(np.mean((p-y)**2)))

def oof_frame(rows):
    df = pd.DataFrame(rows)
    y, p = _validated_binary_inputs(df['label'], df['probability'])
    df['label'] = y; df['probability'] = p
    df['prediction'] = (df['probability'] >= 0.5).astype(int)
    df['correct'] = (df['prediction'] == df['label']).astype(int)
    df['confidence'] = np.maximum(df['probability'], 1-df['probability'])
    df['nll'] = -(df['label']*np.log(np.clip(df['probability'],1e-7,1)) +
                  (1-df['label'])*np.log(np.clip(1-df['probability'],1e-7,1)))
    # Strict metadata lookup: missing subjects or fields raise KeyError.
    for col in ('study','diagnosis','hy','hy_label','hy_status','age','gender'):
        df[col] = df['subject'].map(lambda s: clinical[s][col])
    return df

def participant_oof(df, group_cols):
    """Equal weight per participant; probability is averaged over that person's walks."""
    keys = group_cols + ['subject']
    df = df.copy()
    y, p = _validated_binary_inputs(df['label'], df['probability'])
    df['label'] = y; df['probability'] = p
    if df[keys + ['fold']].isna().any().any():
        raise ValueError('Participant grouping keys and folds must not be missing.')
    grouped = df.groupby(keys, as_index=False)
    counts = grouped[['label', 'fold']].nunique(dropna=False)
    if (counts['label'] > 1).any():
        raise ValueError('Conflicting labels for a subject within an evaluation group.')
    if (counts['fold'] > 1).any():
        raise ValueError('Multiple folds for a subject within an evaluation group.')
    if 'file' in df.columns and df.duplicated(subset=group_cols + ['file']).any():
        raise ValueError('Duplicate file within an evaluation group.')
    agg = dict(label=('label','first'), fold=('fold','first'), probability=('probability','mean'),
               recordings=('file','nunique') if 'file' in df.columns else ('subject','size'))
    out = grouped.agg(**agg)
    return oof_frame(out.to_dict('records'))

def summarize_oof(df, group_cols):
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols, keys))
        rec = metric_dict(g['label'], g['probability'])
        part = participant_oof(g, group_cols)
        pm = metric_dict(part['label'], part['probability'])
        rows.append({**base, **{'recording_'+k:v for k,v in rec.items()}})
        rows[-1].update({'participant_'+k:v for k,v in pm.items()})
    return pd.DataFrame(rows)

def wilson_interval(correct, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = correct/n; den = 1+z*z/n
    center = (ph+z*z/(2*n))/den
    half = z*np.sqrt(ph*(1-ph)/n+z*z/(4*n*n))/den
    return center-half, center+half

def error_tables(df, group_cols):
    part = participant_oof(df, group_cols)
    errors = part[part['correct'] == 0].sort_values(group_cols+['nll'], ascending=[True]*len(group_cols)+[False])
    pd_rows = []
    for keys, g in part[part['label'] == 1].groupby(group_cols+['hy_label'], dropna=False):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols+['hy_label'], keys))
        n = len(g); correct = int(g['correct'].sum()); lo, hi = wilson_interval(correct, n)
        pd_rows.append({**base, 'pd_participants':n, 'correct':correct, 'errors':n-correct,
                        'pd_sensitivity':correct/n, 'wilson_95_low':lo, 'wilson_95_high':hi})
    study_rows = []
    for keys, g in part.groupby(group_cols+['study','diagnosis']):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols+['study','diagnosis'], keys))
        study_rows.append({**base, **metric_dict(g['label'], g['probability'])})
    return part, errors, pd.DataFrame(pd_rows), pd.DataFrame(study_rows)

def show_table(df, title, limit=100):
    print('\n'+title)
    display(df.head(limit))

def pooled_bootstrap(df, group_cols, metrics=('accuracy','balanced_accuracy','macro_f1','sensitivity','specificity','roc_auc')):
    """Stratified participant bootstrap conditional on fitted OOF predictions."""
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols, keys))
        part = participant_oof(g, group_cols).reset_index(drop=True)
        y = part['label'].to_numpy(); p = part['probability'].to_numpy()
        class_idx = [np.flatnonzero(y == c) for c in (0,1)]
        rng = np.random.default_rng(2026)
        for metric in metrics:
            observed = metric_dict(y,p)[metric]
            vals = []
            for _ in range(N_BOOT):
                ii = np.concatenate([rng.choice(ix, len(ix), replace=True) for ix in class_idx])
                vals.append(metric_dict(y[ii],p[ii])[metric])
            rows.append({**base,'metric':metric,'estimate':observed,
                         'bootstrap_95_low':np.nanpercentile(vals,2.5),
                         'bootstrap_95_high':np.nanpercentile(vals,97.5)})
    return pd.DataFrame(rows)

def plot_confusion_and_roc(df, group_cols, title_col):
    part = participant_oof(df, group_cols)
    groups = list(part.groupby(group_cols))
    fig, axes = plt.subplots(len(groups), 2, figsize=(11,4.3*len(groups)), squeeze=False)
    for row, (keys,g) in enumerate(groups):
        if not isinstance(keys, tuple): keys=(keys,)
        title = ' | '.join(f'{c}={v}' for c,v in zip(group_cols,keys))
        cm = confusion_matrix(g['label'],g['prediction'],labels=[0,1])
        ConfusionMatrixDisplay(cm,display_labels=['Control','PD']).plot(ax=axes[row,0],colorbar=False)
        axes[row,0].set_title(title+' | participant OOF')
        fpr,tpr,_ = roc_curve(g['label'],g['probability'])
        axes[row,1].plot(fpr,tpr,label=f"AUC={roc_auc_score(g['label'],g['probability']):.3f}")
        axes[row,1].plot([0,1],[0,1],'--',color='gray')
        axes[row,1].set(xlabel='False-positive rate',ylabel='True-positive rate',title=title)
        axes[row,1].legend()
    fig.tight_layout(); plt.show()


In [ ]:
def build_variantA(**kwargs):
    return nb.build_variantA(**kwargs)

SPACE_A=dict(channels=[32,48],kernels=[(7,5,3),(9,7,5)],se_ratio=[4,8],
             spat_drop=[.1,.2,.3],drop=[.25,.4],l2=[1e-4,1e-3])

def parameter_report(cfg):
    return nb.model_parameter_report(FAMILY,cfg,WIN,LP_ORDER)

MODEL_READY=False
smoke_cfg={k:v[0] for k,v in SPACE_A.items()}
print('Actual forward/backward and safe model/optimizer reload:',nb.smoke_check(FAMILY,smoke_cfg,WIN))
MODEL_READY=True
print('Attention pooling uses a registered WeightedSum layer instead of an anonymous Lambda.')

Actual forward/backward and safe model/optimizer reload: {'family': 'A', 'parameters': 8032, 'tensorflow': '2.21.0', 'keras': '3.13.2', 'safe_model_optimizer_roundtrip': True, 'padding_invariance': False}
Attention pooling uses a registered WeightedSum layer instead of an anonymous Lambda.


In [ ]:
# Apply AFTER original configuration and SPACE_A definitions, BEFORE storage/training.
RUN_PROFILE='full'  # Options: efficient, full, fixed. Changes need a new RUN_ID.
REQUIRE_GPU=True
_PROFILE_KEYS=('SOURCE_REV','FAMILY','SEED','FOLD_SEED','SPLIT_SEED','SEARCH_SEED','N_FOLDS','N_INNER',
    'VAL_FRAC','LP_ORDER','WIN','STRIDE','BATCH_WIN','BATCH_REC','PRED_BATCH','LR_WIN','LR_REC',
    'LABEL_SMOOTH','L2_REG','MAX_EPOCHS_WIN','MAX_EPOCHS_REC','MAX_EPOCHS_MIL','PATIENCE',
    'RLR_PATIENCE','MIN_LR','TRAIN_SEEDS','N_TRIALS','THRESHOLD','COHORT_POLICY','SPACE_A',
    'SPACE_RESMIL','EXPORT_ATTENTION','RUN_PROFILE','CACHE_FROZEN_ENCODER','CONDITIONS')
def scientific_profile(namespace):
    return nb.json_ready({k:namespace[k] for k in _PROFILE_KEYS if k in namespace})
_proposed=scientific_profile({**globals(),**nb.profile_settings(FAMILY,RUN_PROFILE)})
if 'run_result' in globals() and 'FROZEN_SCIENTIFIC_PROFILE' not in globals():
    raise RuntimeError('Existing run_result has no frozen profile. Use a fresh kernel and new checkpoint RUN_ID; do not relabel results.')
if 'FROZEN_SCIENTIFIC_PROFILE' in globals() and _proposed!=globals()['FROZEN_SCIENTIFIC_PROFILE']:
    raise RuntimeError('Scientific profile changed. Start a fresh kernel with a new checkpoint RUN_ID.')
globals().update(nb.profile_settings(FAMILY,RUN_PROFILE))
FROZEN_SCIENTIFIC_PROFILE=scientific_profile(globals())
print('Frozen profile:',RUN_PROFILE)
print('Budget:',nb.budget_report(globals(),FAMILY))
print('Efficient: 10 folds, 1 seed, 3 candidates; epoch caps 100/40; stopping/LR patience 12/5.')
print('Fixed: one configuration, no redundant inner configuration assessment; retain epoch selection and refit.')
print('Cached frozen encoders run in inference mode: frozen noise/dropout is disabled, unlike the full original protocol; training distributions are not identical.')

Frozen profile: full
Budget: {'outer_tasks': 10, 'two_stage_fits': 380, 'training_stages': 760, 'maximum_total_epochs': 456000}
Efficient: 10 folds, 1 seed, 3 candidates; epoch caps 100/40; stopping/LR patience 12/5.
Fixed: one configuration, no redundant inner configuration assessment; retain epoch selection and refit.
Cached frozen encoders run in inference mode: frozen noise/dropout is disabled, unlike the full original protocol; training distributions are not identical.


In [ ]:
# Corrected selection: early-stopping controls are excluded from LP fitting.
# This applies both to inner configuration assessment and outer-development epoch selection.
# Only after selection are LP and both stages refitted on all outer-training participants.
# Candidate lists and inner partitions are independent of training seed.
preview=nb.unique_candidates(SPACE_A,N_TRIALS,SEARCH_SEED)
show_table(pd.DataFrame(preview),'Outer-fold-0 unique candidate preview (not model results)')
print('This protocol change requires a new checkpoint directory; old CSV-only checkpoints cannot be mixed in.')


Outer-fold-0 unique candidate preview (not model results)


,channels,kernels,se_ratio,spat_drop,drop,l2
0,48,"(7, 5, 3)",4,0.3,0.25,0.0010
1,48,"(9, 7, 5)",8,0.3,0.40,0.0001
2,48,"(7, 5, 3)",4,0.2,0.40,0.0010
3,32,"(9, 7, 5)",4,0.2,0.25,0.0001
4,48,"(9, 7, 5)",8,0.2,0.40,0.0010
5,48,"(7, 5, 3)",8,0.2,0.25,0.0010
6,32,"(9, 7, 5)",4,0.1,0.25,0.0001
7,48,"(9, 7, 5)",8,0.3,0.25,0.0010
8,32,"(7, 5, 3)",8,0.1,0.25,0.0010
9,32,"(9, 7, 5)",8,0.2,0.25,0.0001


This protocol change requires a new checkpoint directory; old CSV-only checkpoints cannot be mixed in.


In [ ]:
# REQUIRED preflight, even when RUN_EXPERIMENT=False. This cell never starts training.
RUN_ID='A_efficient_v3'  # Unique per scientific plan; one runtime/writer per ID.
STORAGE_BACKEND='colab_drive'
DRIVE_ROOT='/content/drive/MyDrive/gait-checkpoints'
RCLONE_REMOTE=''  # Kaggle: choose 'rclone' and a real authenticated remote, e.g. researchdrive:gait-checkpoints.
# Configure rclone authentication outside notebook cells; never paste credentials or tokens here.
REQUIRE_DURABLE_STORAGE=True
if RESTORE_FROM_ZIP:
    raise RuntimeError('Leave RESTORE_FROM_ZIP empty. Durable storage restores automatically; do not mix manual ZIP state.')
_storage_settings=(STORAGE_BACKEND,DRIVE_ROOT,RCLONE_REMOTE)
if globals().get('DURABLE_RUN_ID')!=RUN_ID or globals().get('DURABLE_MIRROR') is None:
    if 'run_result' in globals():
        raise RuntimeError('Existing run_result cannot be rebound to another mirror. Restart the kernel; use the same RUN_ID to resume or a new ID for a new plan.')
    CHECKPOINT_DIR=Path(tempfile.mkdtemp(prefix='gait-scratch-'))
    DURABLE_MIRROR=nb.connect_storage(CHECKPOINT_DIR,RUN_ID,backend=STORAGE_BACKEND,
        drive_root=DRIVE_ROOT,rclone_remote=RCLONE_REMOTE)
    DURABLE_RUN_ID=RUN_ID
    DURABLE_STORAGE_SETTINGS=_storage_settings
else:
    DURABLE_MIRROR=globals()['DURABLE_MIRROR']
    CHECKPOINT_DIR=Path(globals()['CHECKPOINT_DIR'])
    if globals().get('DURABLE_STORAGE_SETTINGS')!=_storage_settings:
        raise RuntimeError('Storage settings changed for this RUN_ID. Restart and reconnect explicitly; do not mix backends.')
    if DURABLE_MIRROR.local_root!=CHECKPOINT_DIR.resolve():
        raise RuntimeError('Mirror does not match CHECKPOINT_DIR. Restart and restore into fresh scratch.')
    DURABLE_MIRROR.verified=False  # A failed re-probe must not leave a stale success flag.
    DURABLE_MIRROR.verify()  # Repeat upload/download probe, without overwriting local progress.
if not DURABLE_MIRROR.verified:
    raise RuntimeError('Storage verification failed; training is disabled.')
OUTDIR=CHECKPOINT_DIR/'reports'
print('Verified durable run:',DURABLE_RUN_ID,'| local scratch:',CHECKPOINT_DIR)
print('Automatic restore complete. Epoch commits are verified off-runtime; cloud failures stop training. No training started.')

Mounted at /content/drive
Restored 707 verified checkpoint units from durable storage.
Verified durable run: A_efficient_v3 | local scratch: /tmp/gait-scratch-dvt2hsps
Automatic restore complete. Epoch commits are verified off-runtime; cloud failures stop training. No training started.


In [13]:
if RUN_EXPERIMENT:
    assert MODEL_READY,'Run the real model/checkpoint smoke check first'
    if scientific_profile(globals())!=FROZEN_SCIENTIFIC_PROFILE:
        raise RuntimeError('Frozen scientific profile changed; use a fresh kernel and new RUN_ID.')
    if RESTORE_FROM_ZIP:
        raise RuntimeError('Do not use manual ZIP restore; the storage cell restores committed state automatically.')
    assert REQUIRE_DURABLE_STORAGE and REQUIRE_GPU,'Keep storage and GPU requirements enabled'
    assert DURABLE_RUN_ID==RUN_ID and DURABLE_STORAGE_SETTINGS==(STORAGE_BACKEND,DRIVE_ROOT,RCLONE_REMOTE),'Rerun storage preflight'
    assert DURABLE_MIRROR.verified and DURABLE_MIRROR.local_root==Path(CHECKPOINT_DIR).resolve(),'Verify the matching storage mirror first'
    experiment=nb.NotebookExperiment(globals(),FAMILY)  # Engine consumes DURABLE_MIRROR and verifies requirements.
    run_result=experiment.nested(max_tasks=MAX_NEW_TASKS)
    globals().update(nb.unpack_result(globals(),run_result,FAMILY))
    print('Completed outer tasks:',len(run_result['selections']),'/',len(TRAIN_SEEDS)*N_FOLDS)
    print('Experiment complete:',EXPERIMENT_COMPLETE)
else:
    EXPERIMENT_COMPLETE=False
    print('Training disabled. Verify storage even now; set RUN_EXPERIMENT=True only after profile and preflight checks.')

Streaming output truncated to the last 5000 lines.
  epoch 26: loss=0.4482, monitored=0.5025, train=0.2s val=0.0s remote=0.0s
DURABLE: epoch/tasks/602d7e4bba8fc788bcd095e1f0d94b35f7963a83cbf4f5f56b0c46c79bf22556/recording revision 27 verified (0.0s)
  epoch 27: loss=0.4643, monitored=0.4989, train=0.2s val=0.0s remote=0.1s
DURABLE: epoch/tasks/602d7e4bba8fc788bcd095e1f0d94b35f7963a83cbf4f5f56b0c46c79bf22556/recording revision 28 verified (0.0s)
  epoch 28: loss=0.4633, monitored=0.4954, train=0.2s val=0.0s remote=0.0s
DURABLE: epoch/tasks/602d7e4bba8fc788bcd095e1f0d94b35f7963a83cbf4f5f56b0c46c79bf22556/recording revision 29 verified (0.0s)
  epoch 29: loss=0.4316, monitored=0.4920, train=0.2s val=0.0s remote=0.0s
DURABLE: epoch/tasks/602d7e4bba8fc788bcd095e1f0d94b35f7963a83cbf4f5f56b0c46c79bf22556/recording revision 30 verified (0.0s)
  epoch 30: loss=0.4475, monitored=0.4887, train=0.2s val=0.0s remote=0.0s
DURABLE: epoch/tasks/602d7e4bba8fc788bcd095e1f0d94b35f7963a83cbf4f5f56b0c46c79

In [14]:
if EXPERIMENT_COMPLETE:
    globals().update(nb.summary_tables(globals(),FAMILY))
    show_table(summary.round(4),'Complete pooled recording and participant OOF metrics by seed')
    show_table(participant_ci.round(4),'Participant-stratified bootstrap intervals; conditional on fitted predictions')
    show_table(participant_fold_metrics.round(4),'Participant fold metrics; fold SD is descriptive')
    show_table(selection_table,'Selected Variant-A configurations',limit=100)
    show_table(trial_table,'All inner candidate assessments',limit=100)
else:
    print('Complete all outer tasks before inspecting test performance.')


Complete pooled recording and participant OOF metrics by seed


,family,seed,recording_n,recording_accuracy,recording_balanced_accuracy,recording_macro_f1,recording_f1_pd,recording_sensitivity,recording_specificity,recording_roc_auc,...,participant_n,participant_accuracy,participant_balanced_accuracy,participant_macro_f1,participant_f1_pd,participant_sensitivity,participant_specificity,participant_roc_auc,participant_log_loss,participant_brier
0,A,42,292,0.8425,0.7954,0.8078,0.8894,0.9204,0.6703,0.9005,...,161,0.8012,0.7895,0.7936,0.8333,0.8889,0.6901,0.8865,0.4329,0.1387



Participant-stratified bootstrap intervals; conditional on fitted predictions


,family,seed,metric,estimate,bootstrap_95_low,bootstrap_95_high
0,A,42,accuracy,0.8012,0.7391,0.8634
1,A,42,balanced_accuracy,0.7895,0.7265,0.8444
2,A,42,macro_f1,0.7936,0.7267,0.8527
3,A,42,sensitivity,0.8889,0.8222,0.9444
4,A,42,specificity,0.6901,0.5775,0.7887
5,A,42,roc_auc,0.8865,0.8319,0.9341



Participant fold metrics; fold SD is descriptive


,family,seed,fold,n,accuracy,balanced_accuracy,macro_f1,f1_pd,sensitivity,specificity,roc_auc,log_loss,brier
0,A,42,0,17,0.8235,0.8194,0.8211,0.8421,0.8889,0.7500,0.9583,0.4130,0.1318
1,A,42,1,16,0.7500,0.7460,0.7460,0.7778,0.7778,0.7143,0.7937,0.5284,0.1642
2,A,42,2,16,0.8750,0.8571,0.8667,0.9000,1.0000,0.7143,0.9048,0.3671,0.1129
3,A,42,3,16,0.8125,0.8016,0.8057,0.8421,0.8889,0.7143,0.8730,0.4922,0.1502
4,A,42,4,16,0.8125,0.7857,0.7922,0.8571,1.0000,0.5714,0.9683,0.3843,0.1195
5,A,42,5,16,0.8125,0.8175,0.8118,0.8235,0.7778,0.8571,0.9048,0.4200,0.1315
6,A,42,6,16,0.8125,0.8016,0.8057,0.8421,0.8889,0.7143,0.9206,0.4077,0.1410
7,A,42,7,16,0.7500,0.7302,0.7333,0.8000,0.8889,0.5714,0.8571,0.4338,0.1421
8,A,42,8,16,0.8125,0.7857,0.7922,0.8571,1.0000,0.5714,0.9683,0.4460,0.1434
9,A,42,9,16,0.7500,0.7460,0.7460,0.7778,0.7778,0.7143,0.8730,0.4382,0.1511



Selected Variant-A configurations


,best_recording_epoch,best_window_epoch,candidate,classifier_parameters,family,inner_participant_accuracy,inner_participant_auc,inner_participant_brier,inner_participant_log_loss,mean_best_recording_epoch,mean_best_window_epoch,outer_fold,predictor_parameters,seed,total_parameters,trial
0,108,3,"{'channels': 32, 'drop': 0.25, 'kernels': [9, ...",6644,A,0.729167,0.827356,0.182780,0.545718,67.666667,6.666667,0,198,42,6842,9
1,200,6,"{'channels': 48, 'drop': 0.25, 'kernels': [9, ...",16820,A,0.848276,0.908372,0.114747,0.380290,138.333333,20.333333,1,198,42,17018,9
2,121,11,"{'channels': 32, 'drop': 0.4, 'kernels': [7, 5...",8032,A,0.758621,0.836227,0.166234,0.498897,102.000000,12.000000,2,198,42,8230,11
3,78,4,"{'channels': 32, 'drop': 0.25, 'kernels': [9, ...",8196,A,0.806897,0.878858,0.141481,0.433919,152.333333,20.000000,3,198,42,8394,6
4,50,11,"{'channels': 48, 'drop': 0.25, 'kernels': [7, ...",16592,A,0.772414,0.833526,0.167069,0.506494,150.666667,12.333333,4,198,42,16790,8
5,82,4,"{'channels': 32, 'drop': 0.4, 'kernels': [7, 5...",8032,A,0.786207,0.839506,0.162904,0.503773,120.666667,23.000000,5,198,42,8230,6
6,200,21,"{'channels': 32, 'kernels': (9, 7, 5), 'se_rat...",8196,A,0.806897,0.846451,0.151446,0.480698,46.000000,30.000000,6,198,42,8394,10
7,104,12,"{'channels': 32, 'kernels': (7, 5, 3), 'se_rat...",8032,A,0.793103,0.859182,0.152211,0.493450,98.666667,16.333333,7,198,42,8230,8
8,84,5,"{'channels': 32, 'kernels': (9, 7, 5), 'se_rat...",6644,A,0.737931,0.824460,0.181126,0.545511,103.000000,33.666667,8,198,42,6842,2
9,168,23,"{'channels': 48, 'kernels': (7, 5, 3), 'se_rat...",13112,A,0.751724,0.861111,0.154800,0.469868,67.333333,15.333333,9,198,42,13310,8



All inner candidate assessments


,candidate,classifier_parameters,family,inner_participant_accuracy,inner_participant_auc,inner_participant_brier,inner_participant_log_loss,mean_best_recording_epoch,mean_best_window_epoch,outer_fold,predictor_parameters,seed,total_parameters,trial
0,"{'channels': 48, 'drop': 0.25, 'kernels': [7, ...",16592,A,0.708333,0.800705,0.190687,0.566190,40.666667,28.666667,0,198,42,16790,0
1,"{'channels': 48, 'drop': 0.4, 'kernels': [9, 7...",13340,A,0.694444,0.804429,0.201268,0.588429,52.000000,6.333333,0,198,42,13538,1
2,"{'channels': 48, 'drop': 0.4, 'kernels': [7, 5...",16592,A,0.701389,0.789536,0.194258,0.574364,38.666667,25.000000,0,198,42,16790,2
3,"{'channels': 32, 'drop': 0.25, 'kernels': [9, ...",8196,A,0.736111,0.816187,0.187418,0.554409,62.333333,6.666667,0,198,42,8394,3
4,"{'channels': 48, 'drop': 0.4, 'kernels': [9, 7...",13340,A,0.680556,0.781305,0.199507,0.588344,42.000000,35.666667,0,198,42,13538,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,"{'channels': 48, 'kernels': (7, 5, 3), 'se_rat...",13112,A,0.786207,0.826003,0.163848,0.524061,113.666667,28.333333,7,198,42,13310,11
96,"{'channels': 32, 'kernels': (7, 5, 3), 'se_rat...",8032,A,0.648276,0.744020,0.216257,0.624719,93.333333,14.000000,8,198,42,8230,0
97,"{'channels': 32, 'kernels': (9, 7, 5), 'se_rat...",8196,A,0.689655,0.748843,0.213884,0.624213,99.666667,6.666667,8,198,42,8394,1
98,"{'channels': 32, 'kernels': (9, 7, 5), 'se_rat...",6644,A,0.737931,0.824460,0.181126,0.545511,103.000000,33.666667,8,198,42,6842,2


In [15]:
if EXPERIMENT_COMPLETE:
    globals().update(nb.error_reports(globals(),FAMILY))
    show_table(participants.sort_values('nll',ascending=False).round(4),'Complete participant table, highest individual NLL first',limit=200)
    show_table(participant_errors.round(4),'Misclassified participants',limit=200)
    show_table(pd_severity.round(4),'PD sensitivity by verified HY stage')
    show_table(by_study.round(4),'Study-by-diagnosis metrics; single-class AUC remains undefined')
    show_table(seed_stability.round(4),'Across-seed errors and probability variability')
    show_table(inner_loss.head(50).round(4),'Selected inner-assessment NLL contributors; repetitions across outer folds are not independent')
else:
    print('Error diagnostics wait for the complete experiment.')


Complete participant table, highest individual NLL first


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains cl

,family,seed,subject,label,fold,probability,recordings,prediction,correct,confidence,nll,study,diagnosis,hy,hy_label,hy_status,age,gender
55,A,42,JuCo12,0,1,0.9587,1,1,0,0.9587,3.1876,Ju,Control,NaN,Not applicable,not_applicable,72.0,2
138,A,42,SiPt17,1,3,0.0584,1,0,0,0.9416,2.8409,Si,PD,2.0,2.0,verified,64.0,1
63,A,42,JuCo20,0,2,0.8997,1,1,0,0.8997,2.2992,Ju,Control,NaN,Not applicable,not_applicable,67.0,2
54,A,42,JuCo11,0,6,0.8748,1,1,0,0.8748,2.0780,Ju,Control,NaN,Not applicable,not_applicable,72.0,1
155,A,42,SiPt35,1,9,0.1397,1,0,0,0.8603,1.9686,Si,PD,2.0,2.0,verified,57.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44,A,42,GaPt33,1,9,0.9854,3,1,1,0.9854,0.0147,Ga,PD,2.5,2.5,verified,63.0,1
136,A,42,SiPt15,1,9,0.9876,1,1,1,0.9876,0.0125,Si,PD,2.0,2.0,verified,62.0,1
99,A,42,SiCo04,0,2,0.0076,1,0,1,0.9924,0.0076,Si,Control,NaN,Not applicable,not_applicable,71.0,1
125,A,42,SiCo30,0,6,0.0058,1,0,1,0.9942,0.0058,Si,Control,NaN,Not applicable,not_applicable,63.0,1



Misclassified participants


,family,seed,subject,label,fold,probability,recordings,prediction,correct,confidence,nll,study,diagnosis,hy,hy_label,hy_status,age,gender
55,A,42,JuCo12,0,1,0.9587,1,1,0,0.9587,3.1876,Ju,Control,NaN,Not applicable,not_applicable,72.0,2
138,A,42,SiPt17,1,3,0.0584,1,0,0,0.9416,2.8409,Si,PD,2.0,2.0,verified,64.0,1
63,A,42,JuCo20,0,2,0.8997,1,1,0,0.8997,2.2992,Ju,Control,NaN,Not applicable,not_applicable,67.0,2
54,A,42,JuCo11,0,6,0.8748,1,1,0,0.8748,2.0780,Ju,Control,NaN,Not applicable,not_applicable,72.0,1
155,A,42,SiPt35,1,9,0.1397,1,0,0,0.8603,1.9686,Si,PD,2.0,2.0,verified,57.0,1
112,A,42,SiCo17,0,0,0.8570,1,1,0,0.8570,1.9452,Si,Control,NaN,Not applicable,not_applicable,64.0,2
129,A,42,SiPt07,1,6,0.1673,1,0,0,0.8327,1.7878,Si,PD,2.0,2.0,verified,54.0,2
3,A,42,GaCo04,0,3,0.8318,2,1,0,0.8318,1.7823,Ga,Control,NaN,Not applicable,not_applicable,86.0,1
5,A,42,GaCo06,0,7,0.8304,2,1,0,0.8304,1.7745,Ga,Control,NaN,Not applicable,not_applicable,82.0,1
150,A,42,SiPt30,1,5,0.1703,1,0,0,0.8297,1.7703,Si,PD,2.0,2.0,verified,60.0,1



PD sensitivity by verified HY stage


... (3 outputs left out)


Across-seed errors and probability variability


,family,subject,error_seeds,n_seeds,probability_mean,probability_std,prediction_variants
0,A,GaCo01,1,1,0.5122,NaN,1
1,A,GaCo02,0,1,0.0753,NaN,1
2,A,GaCo03,0,1,0.0049,NaN,1
3,A,GaCo04,1,1,0.8318,NaN,1
4,A,GaCo05,1,1,0.6297,NaN,1
...,...,...,...,...,...,...,...
95,A,JuPt28,0,1,0.9627,NaN,1
96,A,JuPt29,0,1,0.7641,NaN,1
97,A,SiCo01,1,1,0.7870,NaN,1
98,A,SiCo03,0,1,0.3585,NaN,1



Selected inner-assessment NLL contributors; repetitions across outer folds are not independent


,family,seed,outer_fold,trial,subject,label,fold,probability,recordings,prediction,correct,confidence,nll,study,diagnosis,hy,hy_label,hy_status,age,gender
1137,A,42,7,8,SiPt17,1,0,0.0078,1,0,0,0.9922,4.8518,Si,PD,2.0,2.0,verified,64.0,1
270,A,42,1,9,SiPt17,1,0,0.0303,1,0,0,0.9697,3.4977,Si,PD,2.0,2.0,verified,64.0,1
1151,A,42,7,8,SiPt32,1,0,0.0406,1,0,0,0.9594,3.2048,Si,PD,2.0,2.0,verified,64.0,1
1210,A,42,8,2,JuCo12,0,0,0.9593,1,1,0,0.9593,3.2025,Ju,Control,NaN,Not applicable,not_applicable,72.0,2
689,A,42,4,8,SiCo28,0,1,0.9535,1,1,0,0.9535,3.0685,Si,Control,NaN,Not applicable,not_applicable,61.0,1
917,A,42,6,10,JuCo12,0,1,0.9500,1,1,0,0.9500,2.9967,Ju,Control,NaN,Not applicable,not_applicable,72.0,2
280,A,42,1,9,SiPt30,1,1,0.0524,1,0,0,0.9476,2.9495,Si,PD,2.0,2.0,verified,60.0,1
204,A,42,1,9,JuCo20,0,1,0.9448,1,1,0,0.9448,2.8964,Ju,Control,NaN,Not applicable,not_applicable,67.0,2
1069,A,42,7,8,JuCo20,0,0,0.9436,1,1,0,0.9436,2.8761,Ju,Control,NaN,Not applicable,not_applicable,67.0,2
678,A,42,4,8,SiCo17,0,0,0.9381,1,1,0,0.9381,2.7826,Si,Control,NaN,Not applicable,not_applicable,64.0,2


In [17]:
# Optional portability/report exports; verified remote checkpoints already provide automatic recovery.
EXPORT_CHECKPOINT_BUNDLE=False
EXPORT_ANALYSIS_CSV=True
if EXPORT_CHECKPOINT_BUNDLE:
    print('Optional checkpoint ZIP:',nb.export_checkpoint_bundle(CHECKPOINT_DIR))
    print('Download this optional copy if wanted; do not mix manual ZIP restore with the durable runner.')
if EXPORT_ANALYSIS_CSV:
    if not EXPERIMENT_COMPLETE: raise RuntimeError('Analysis export requires complete OOF results')
    nb.export_tables(globals(),FAMILY)
if not EXPORT_CHECKPOINT_BUNDLE and not EXPORT_ANALYSIS_CSV:
    print('No extra export requested. Training epoch/task state is committed automatically to verified off-runtime storage.')

Analysis tables: /tmp/gait-scratch-dvt2hsps/reports


## Reporting checklist

Report the frozen profile, all outer folds/seeds, candidate budget, curated-cohort exclusions and verified clinical coverage. Nested assessment is distinct from early stopping: neither early-stopping nor inner-assessment controls fit the selection-stage LP reference. The final fixed-epoch refit uses all outer-training participants only. The `fixed` profile omits configuration assessment, not epoch selection or refitting.

The Keras 3 version fixes earlier early-control preprocessing exposure and seed-dependent search partitions. Use a new RUN_ID and do not combine OOF rows with older results. Report modern separable-convolution regularizers and the inference-mode frozen-cache protocol; do not claim bit-identical historical training or identical stochastic training distributions.

Verified remote checkpoint state is authoritative, not exported CSV row counts. Per-epoch commits preserve model/optimizer, history and stopping/LR counters; inner/outer task commits preserve completed work. Scratch loss is safe after verified commits; a mid-epoch crash can lose the current epoch. Cloud failure stops training; neither permanent provider availability nor bitwise equivalence across GPU runtimes is guaranteed. Automatic restore replaces the old manual ZIP recovery workflow; optional exports remain useful for portable reports.

Participant bootstrap intervals condition on fitted OOF predictions, not new clinics or model-refitting uncertainty. HY/study subgroups are descriptive. The update preserves smoke checks and diagnostics, not new full-cohort measurements.